# Quelle — Training on Colab

Disconnect-resistant training with periodic sync to Google Drive.

**Flow:** Mount Drive → Clone repo → Restore artifacts from Drive → Setup data → Train (syncing to Drive every N minutes) → Final sync.

On reconnect: re-run all cells. The notebook detects the last saved checkpoint and resumes automatically.

---
**Before running:** edit the Configuration cell below.

In [ ]:
# ============================================================
# Configuration — edit this cell
# ============================================================

REPO_URL    = "https://github.com/leonorae/quelle.git"
BRANCH      = "main"

# Scripts to run, relative to repo root
SETUP_SCRIPT = "experiments/VVVVVV/src/setup.sh"
TRAIN_SCRIPT = "experiments/VVVVVV/src/train_d12.sh"

# nanochat model tag — must match --model-tag in TRAIN_SCRIPT
MODEL_TAG = "d12"

# Training parameters (override script defaults)
N_SHARDS     = 30
N_ITERATIONS = 6000

# Drive path where artifacts are persisted
DRIVE_ARTIFACT_DIR = "/content/drive/MyDrive/quelle_artifacts"

# Sync interval in minutes (Drive ← local)
SYNC_INTERVAL_MIN = 5

# Local paths (fast Colab SSD — lost on disconnect, hence the sync)
REPO_DIR         = "/content/quelle"
LOCAL_NANOCHAT   = "/content/nanochat_base"

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Clone / update repo

In [ ]:
import os, subprocess, sys

def run(cmd, **kwargs):
    """Run a shell command, streaming output, raising on failure."""
    print(f"$ {' '.join(cmd) if isinstance(cmd, list) else cmd}")
    result = subprocess.run(cmd, **kwargs)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")

if not os.path.exists(REPO_DIR):
    run(["git", "clone", "--recurse-submodules", REPO_URL, REPO_DIR])
    run(["git", "-C", REPO_DIR, "checkout", BRANCH])
else:
    run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    run(["git", "-C", REPO_DIR, "checkout", BRANCH])
    run(["git", "-C", REPO_DIR, "merge", "--ff-only", f"origin/{BRANCH}"])
    run(["git", "-C", REPO_DIR, "submodule", "update", "--init"])

NANOCHAT_DIR = os.path.join(REPO_DIR, "nanochat")
print(f"Repo: {REPO_DIR}")
print(f"nanochat: {NANOCHAT_DIR}")

## 3. Install uv + sync dependencies

In [ ]:
# Fix: Colab's PyTorch requires libcusparseLt.so.0 from the system CUDA libs,
# but they aren't on LD_LIBRARY_PATH by default.  Add them now so every
# subprocess that imports torch (uv sync, setup, train) can find the library.
_cuda_lib = "/usr/local/cuda/lib64"
if os.path.exists(_cuda_lib) and _cuda_lib not in os.environ.get("LD_LIBRARY_PATH", ""):
    os.environ["LD_LIBRARY_PATH"] = _cuda_lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    print(f"Added {_cuda_lib} to LD_LIBRARY_PATH")

# Detect CUDA to select the right torch extra (gpu → cu128 index, cpu → CPU-only wheel).
# Plain `uv sync` skips both indexes because nanochat gates them behind extras.
_has_cuda = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
TORCH_EXTRA = "gpu" if _has_cuda else "cpu"
print(f"CUDA detected: {_has_cuda} → using --extra {TORCH_EXTRA}")
if not _has_cuda:
    print("WARNING: No GPU found. Training will be extremely slow.")
    print("         In Colab: Runtime → Change runtime type → GPU")

# Install uv if not present
if subprocess.run(["which", "uv"], capture_output=True).returncode != 0:
    run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True)
    os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

run(["uv", "sync", "--extra", TORCH_EXTRA], cwd=NANOCHAT_DIR)

# Verify torch sees the GPU
_check = subprocess.run(
    ["uv", "run", "python", "-c",
     "import torch; print(f'torch {torch.__version__}  cuda_available={torch.cuda.is_available()}')"],
    cwd=NANOCHAT_DIR, capture_output=True, text=True
)
print(_check.stdout.strip())
if _has_cuda and "cuda_available=False" in _check.stdout:
    raise RuntimeError("torch installed but CUDA not available — check driver / CUDA version.")

## 4. Restore artifacts from Drive

In [ ]:
import shutil
from pathlib import Path

os.makedirs(LOCAL_NANOCHAT, exist_ok=True)
os.makedirs(DRIVE_ARTIFACT_DIR, exist_ok=True)

drive_contents = list(Path(DRIVE_ARTIFACT_DIR).iterdir())
if drive_contents:
    print(f"Restoring artifacts from Drive ({len(drive_contents)} items at top level)...")
    run(["rsync", "-a", "--info=progress2",
         f"{DRIVE_ARTIFACT_DIR}/", f"{LOCAL_NANOCHAT}/"])
else:
    print("No existing artifacts on Drive — fresh start.")

# Check for an existing checkpoint to resume from
checkpoint_dir = Path(LOCAL_NANOCHAT) / "base_checkpoints" / MODEL_TAG
resume_step = -1
if checkpoint_dir.exists():
    model_files = sorted(checkpoint_dir.glob("model_*.pt"))
    if model_files:
        resume_step = int(model_files[-1].stem.split("_")[-1])
        print(f"Found checkpoint at step {resume_step} — will resume.")

os.environ["NANOCHAT_BASE_DIR"] = LOCAL_NANOCHAT
print(f"NANOCHAT_BASE_DIR={LOCAL_NANOCHAT}")

## 5. Setup: download data shards + train tokenizer

In [ ]:
# Check if tokenizer already exists (setup is idempotent but slow to re-run)
import glob
tokenizer_exists = bool(glob.glob(os.path.join(LOCAL_NANOCHAT, "tokenizer", "*.json"))
                        or glob.glob(os.path.join(LOCAL_NANOCHAT, "tok*.model"))
                        or glob.glob(os.path.join(LOCAL_NANOCHAT, "tokenizer*")))

if tokenizer_exists:
    print("Tokenizer found — skipping setup.")
    # Still ensure enough shards are present
    n_shards = len(glob.glob(os.path.join(LOCAL_NANOCHAT, "base_data_climbmix", "*.parquet")))
    print(f"Data shards present: {n_shards}")
    if n_shards < N_SHARDS:
        print(f"Fewer than {N_SHARDS} shards found — running setup to download more.")
        tokenizer_exists = False

if not tokenizer_exists:
    env = {
        **os.environ,
        "N_SHARDS": str(N_SHARDS),
        "NANOCHAT_BASE_DIR": LOCAL_NANOCHAT,
        "TORCH_EXTRA": TORCH_EXTRA,  # prevent setup.sh's uv sync from switching torch variant
    }
    run(["bash", os.path.join(REPO_DIR, SETUP_SCRIPT)], env=env)

## 6. Start background Drive sync

In [ ]:
import threading, time

_stop_sync = threading.Event()

def _sync_worker():
    while not _stop_sync.wait(timeout=SYNC_INTERVAL_MIN * 60):
        try:
            subprocess.run(
                ["rsync", "-a", f"{LOCAL_NANOCHAT}/", f"{DRIVE_ARTIFACT_DIR}/"],
                capture_output=True
            )
            print(f"[sync] {time.strftime('%H:%M:%S')} → Drive", flush=True)
        except Exception as e:
            print(f"[sync] WARNING: {e}", flush=True)

_sync_thread = threading.Thread(target=_sync_worker, daemon=True)
_sync_thread.start()
print(f"Background sync started (every {SYNC_INTERVAL_MIN} min → {DRIVE_ARTIFACT_DIR})")

## 7. Train

In [ ]:
env = {
    **os.environ,
    "NANOCHAT_BASE_DIR": LOCAL_NANOCHAT,
    "N_ITERATIONS":     str(N_ITERATIONS),
    "RESUME_FROM_STEP": str(resume_step),
}

if resume_step > 0:
    print(f"Resuming from step {resume_step} / {N_ITERATIONS}")
else:
    print(f"Training from scratch for {N_ITERATIONS} steps")

try:
    run(["bash", os.path.join(REPO_DIR, TRAIN_SCRIPT)], env=env, cwd=REPO_DIR)
finally:
    # Always do a final sync, even if training was interrupted
    _stop_sync.set()
    print("\nFinal sync to Drive...")
    run(["rsync", "-a", "--info=progress2",
         f"{LOCAL_NANOCHAT}/", f"{DRIVE_ARTIFACT_DIR}/"])
    print("Done.")

## 8. (Optional) Run Phase 0 probes

Only relevant for experiment VVVVVV. Skip for other experiments.

In [ ]:
PROBE_SCRIPT = "experiments/VVVVVV/src/run_phase0.sh"

env = {**os.environ, "NANOCHAT_BASE_DIR": LOCAL_NANOCHAT}
run(["bash", os.path.join(REPO_DIR, PROBE_SCRIPT)], env=env, cwd=REPO_DIR)

# Sync results to Drive
run(["rsync", "-a", f"{LOCAL_NANOCHAT}/", f"{DRIVE_ARTIFACT_DIR}/"])

# Also sync the results JSON (which lives in the repo outputs/, not NANOCHAT_BASE_DIR)
results_src = os.path.join(REPO_DIR, "experiments/VVVVVV/outputs")
results_dst = os.path.join(DRIVE_ARTIFACT_DIR, "phase0_outputs")
run(["rsync", "-a", f"{results_src}/", f"{results_dst}/"])
print(f"Phase 0 results synced to {results_dst}")